# background

> Estimate and subtract background in Euclid images.

In [ ]:
# | default_exp background

In [ ]:
# | export

from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
from astropy.stats import SigmaClip
from astropy.visualization import AsinhStretch, ImageNormalize
from photutils.background import Background2D, MedianBackground
from scipy.stats import median_abs_deviation

from nicl.main import configure_logging

In [ ]:
# | exporti

configure_logging(__name__)

In [ ]:
# | export

def get_background(image, *, box_size, filter_size, mask=None):
    bkg_estimator = MedianBackground() 
    sigma_clip = SigmaClip(sigma=3.0, maxiters=10)
    bkg = Background2D(image, box_size, mask=mask, filter_size=filter_size, exclude_percentile=90, bkg_estimator=bkg_estimator, sigma_clip=sigma_clip)
    return bkg

In [ ]:
# | export

def plot_background(
    img,  # original image to plot
    bkg,  # background image to plot
    other=None,  # other image to plot
    other_label=None,  # label for the other image
    background_focus=False,
):
    """Create a plot displaying the original image, background and threshold/difference/etc."""
    ncol = 2 if other is None else 3
    fw, fh = plt.rcParams["figure.figsize"]
    fig, ax = plt.subplots(1, ncol, figsize=(2 * fw, fh), layout="constrained")
    rms = median_abs_deviation(img, axis=None, scale="normal", nan_policy="omit")
    median = np.nanmedian(img)
    if background_focus:
        norm = ImageNormalize(vmin=median - 2.5 * rms, vmax=median + 2.5 * rms)
    else:
        norm = ImageNormalize(
            vmin=median - 2.5 * rms, vmax=median + 10 * rms, stretch=AsinhStretch(0.1)
        )
    cmap = mpl.colormaps.get_cmap("viridis")
    cmap.set_bad("black")
    ax[0].imshow(img, norm=norm, origin="lower", cmap=cmap)
    if isinstance(bkg, Background2D):
        ax[1].imshow(bkg.background, norm=norm, origin="lower", interpolation="none", cmap=cmap)
        bkg.plot_meshes(ax=ax[0], outlines=True, marker="+", color="white", alpha=0.75)
    else:
        ax[1].imshow(bkg, norm=norm, origin="lower", interpolation="none", cmap=cmap)
    ax[0].set_title("Original Image")
    ax[1].set_title("Background")
    if other is not None:
        ax[2].imshow(other, norm=norm, origin="lower", interpolation="none", cmap=cmap)
        if other_label is not None:
            ax[2].set_title(other_label)
    for a in ax:
        a.xaxis.set_ticks([])
        a.yaxis.set_ticks([])

In [ ]:
# | export

def make_background_plot(image, bkg, mask=None, output_dir=None, label=None):
    image = np.where(mask, np.nan, image)
    plot_background(image, bkg, image - bkg.background, "Subtracted")
    if output_dir:
        output_dir = Path(output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)
        prefix = f"{label}_" if label else ""
        plt.savefig(output_dir / f"{prefix}background.pdf")
        plt.close()